# Assignment 4

## Apply decision tree on Bank Marketing Data Set. Apply k fold and hold out cross validation techniques.

#### Dataset and objective

The task is to predict whether a client subscribes to a term deposit (`y`). The positive class is relatively uncommon, so sensitivity, specificity, precision, F1-score, and ROC-AUC are reported in addition to accuracy.

In [1]:
from pathlib import Path
from io import BytesIO
from zipfile import ZipFile
from urllib.request import urlopen
import time
import warnings
import numpy as np
import pandas as pd

from scipy.stats import ttest_rel
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
RANDOM_STATE = 42


#### Loading the Bank Marketing dataset

The Bank Marketing data are loaded directly from the UCI Machine Learning Repository. No local cache, OpenML mirror, or alternate data source is used. The UCI archive contains a nested `bank.zip`; the full `bank-full.csv` file is read with its semicolon delimiter.

In [2]:
from io import BytesIO
from zipfile import ZipFile
from urllib.request import urlopen

uci_bank_url = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"
with urlopen(uci_bank_url) as response:
    with ZipFile(BytesIO(response.read())) as archive:
        with archive.open("bank.zip") as nested_file:
            with ZipFile(BytesIO(nested_file.read())) as nested_archive:
                with nested_archive.open("bank-full.csv") as bank_file:
                    bank = pd.read_csv(bank_file, sep=";")
bank = bank.replace("unknown", np.nan)
y = (bank.pop("y").astype(str).str.strip().str.lower() == "yes").astype(int)
X = bank
print("source: UCI Machine Learning Repository — Bank Marketing dataset")
print("shape:", X.shape)
print("class counts:", y.value_counts().sort_index().to_dict())
print("missing values:", int(X.isna().sum().sum()))


source: UCI Machine Learning Repository — Bank Marketing dataset
shape: (45211, 16)
class counts: {0: 39922, 1: 5289}
missing values: 52124


### (a) Report the preprocessing techniques.


1. The target `y` is encoded as 1 for `yes` and 0 for `no`.
2. The dataset's `unknown` category is treated as missing rather than as a genuine category.
3. Numeric variables are median-imputed and standardized.
4. Categorical variables are imputed with their training-fold mode and one-hot encoded.
5. The preprocessing is inside a pipeline, so imputers, category levels, and scaling are fitted only on each training fold. This prevents data leakage.
6. The target distribution is preserved by stratified splitting.

In [3]:
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()
preprocessor = ColumnTransformer([
    ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric_features),
    ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), categorical_features),
], sparse_threshold=0)
print("numeric features:", numeric_features)
print("categorical features:", categorical_features)

numeric features: ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
categorical features: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']


### Decision-tree models

Two tree criteria are compared while holding the other hyperparameters fixed: Gini impurity and entropy/information gain. `min_samples_leaf=20` limits very small leaves and reduces overfitting. `class_weight="balanced"` compensates for the unequal class frequencies using weights computed from the training data.

In [4]:
models = {
    "Decision tree — Gini": Pipeline([
        ("preprocess", preprocessor),
        ("tree", DecisionTreeClassifier(criterion="gini", min_samples_leaf=20, class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
    "Decision tree — Entropy": Pipeline([
        ("preprocess", preprocessor),
        ("tree", DecisionTreeClassifier(criterion="entropy", min_samples_leaf=20, class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
}
scoring = {"accuracy": "accuracy", "sensitivity": "recall", "precision": "precision", "f1": "f1", "roc_auc": "roc_auc"}
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return tn / (tn + fp) if tn + fp else 0.0


#### Validation design

A stratified 80/20 holdout is reserved for final evaluation. On the 80% training portion, stratified 5-fold cross-validation uses the same folds for both tree variants. This makes the fold-wise results paired and supports a paired t-test.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
for name, estimator in models.items():
    result = cross_validate(estimator, X_train, y_train, cv=skf, scoring=scoring, n_jobs=-1)
    for fold in range(5):
        cv_rows.append({"model": name, "fold": fold + 1, **{m: result[f"test_{m}"][fold] for m in scoring}})
cv_results = pd.DataFrame(cv_rows)
cv_summary = cv_results.groupby("model")[list(scoring)].agg(["mean", "std"]).round(4)
cv_summary

accuracy         sensitivity         precision  \
                            mean     std        mean     std      mean   
model                                                                    
Decision tree — Entropy   0.8133  0.0074      0.8135  0.0168    0.3662   
Decision tree — Gini      0.8153  0.0042      0.8216  0.0099    0.3698   

                                     f1         roc_auc          
                            std    mean     std    mean     std  
model                                                            
Decision tree — Entropy  0.0110  0.5049  0.0120  0.8724  0.0064  
Decision tree — Gini     0.0056  0.5099  0.0051  0.8763  0.0042

### (b) Report the value of the performance measures used.

The reported measures are:

- **Accuracy:** fraction of all correct predictions.
- **Sensitivity/recall:** $TP/(TP+FN)$, the fraction of subscribers detected.
- **Specificity:** $TN/(TN+FP)$, the fraction of non-subscribers correctly rejected.
- **Precision:** $TP/(TP+FP)$, the fraction of predicted subscribers who subscribe.
- **F1-score:** harmonic mean of precision and sensitivity.
- **ROC-AUC:** ranking quality over all probability thresholds.

The following table reports both holdout performance and fit time.

In [6]:
def holdout_metrics(y_true, pred, probability):
    return {
        "accuracy": accuracy_score(y_true, pred),
        "sensitivity": recall_score(y_true, pred),
        "specificity": specificity_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred),
        "roc_auc": roc_auc_score(y_true, probability),
    }

holdout_rows = []
fitted = {}
for name, estimator in models.items():
    start = time.perf_counter()
    estimator.fit(X_train, y_train)
    elapsed = time.perf_counter() - start
    pred = estimator.predict(X_test)
    probability = estimator.predict_proba(X_test)[:, 1]
    row = {"model": name, **holdout_metrics(y_test, pred, probability), "fit_time_seconds": elapsed}
    holdout_rows.append(row)
    fitted[name] = estimator
holdout_results = pd.DataFrame(holdout_rows).set_index("model").round(4)
holdout_results

,accuracy,sensitivity,specificity,precision,f1,roc_auc,fit_time_seconds
model,,,,,,,
Decision tree — Gini,0.8185,0.828,0.8173,0.3752,0.5164,0.8697,0.3996
Decision tree — Entropy,0.8068,0.810,0.8064,0.3566,0.4952,0.8687,0.3597


### (c) Compare the performance of the algorithms using t test

For each metric, the two algorithms produce one score on each of the same five folds. A paired t-test tests whether the mean within-fold difference is zero. The null hypothesis is equal mean performance; a p-value below 0.05 is treated as statistically significant evidence against that null. Since only five folds are available, p-values should be interpreted cautiously and reported alongside mean differences and standard deviations.

In [7]:
def paired_t_tests(metric):
    wide = cv_results.pivot(index="fold", columns="model", values=metric)
    names = list(wide.columns)
    difference = wide[names[0]] - wide[names[1]]
    statistic, p_value = ttest_rel(wide[names[0]], wide[names[1]])
    return {
        "metric": metric,
        "first_model": names[0],
        "second_model": names[1],
        "mean_difference": difference.mean(),
        "t_statistic": statistic,
        "p_value": p_value,
    }

t_test_results = pd.DataFrame([paired_t_tests(metric) for metric in scoring])
t_test_results.round(5)

,metric,first_model,second_model,mean_difference,t_statistic,p_value
0,accuracy,Decision tree — Entropy,Decision tree — Gini,-0.00194,-0.59884,0.58154
1,sensitivity,Decision tree — Entropy,Decision tree — Gini,-0.00804,-1.67161,0.16992
2,precision,Decision tree — Entropy,Decision tree — Gini,-0.00358,-0.77924,0.47937
3,f1,Decision tree — Entropy,Decision tree — Gini,-0.00499,-1.09531,0.33490
4,roc_auc,Decision tree — Entropy,Decision tree — Gini,-0.00386,-0.95311,0.39451


#### Interpretation

The best model should not be selected from accuracy alone because subscription is the minority class. The cross-validation mean measures expected performance, the standard deviation measures stability, and the holdout table estimates generalization to unseen data. The paired t-test indicates whether the Gini and entropy differences are statistically distinguishable on the selected folds; a non-significant result means the data do not provide strong evidence that one criterion is superior.

In [8]:
print("Cross-validation summary:")
display(cv_summary)
print("Holdout results:")
display(holdout_results)
print("Paired t-tests:")
display(t_test_results.round(5))

# Visualize the top levels of each fitted tree after preprocessing.
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for axis, (name, estimator) in zip(axes, fitted.items()):
    plot_tree(estimator.named_steps["tree"], max_depth=3, filled=True, ax=axis, node_ids=False, impurity=False, proportion=True)
    axis.set_title(name + " (first 3 levels)")
plt.tight_layout()
plt.show()

Cross-validation summary:


accuracy         sensitivity         precision  \
                            mean     std        mean     std      mean   
model                                                                    
Decision tree — Entropy   0.8133  0.0074      0.8135  0.0168    0.3662   
Decision tree — Gini      0.8153  0.0042      0.8216  0.0099    0.3698   

                                     f1         roc_auc          
                            std    mean     std    mean     std  
model                                                            
Decision tree — Entropy  0.0110  0.5049  0.0120  0.8724  0.0064  
Decision tree — Gini     0.0056  0.5099  0.0051  0.8763  0.0042

Holdout results:


,accuracy,sensitivity,specificity,precision,f1,roc_auc,fit_time_seconds
model,,,,,,,
Decision tree — Gini,0.8185,0.828,0.8173,0.3752,0.5164,0.8697,0.3996
Decision tree — Entropy,0.8068,0.810,0.8064,0.3566,0.4952,0.8687,0.3597


Paired t-tests:


,metric,first_model,second_model,mean_difference,t_statistic,p_value
0,accuracy,Decision tree — Entropy,Decision tree — Gini,-0.00194,-0.59884,0.58154
1,sensitivity,Decision tree — Entropy,Decision tree — Gini,-0.00804,-1.67161,0.16992
2,precision,Decision tree — Entropy,Decision tree — Gini,-0.00358,-0.77924,0.47937
3,f1,Decision tree — Entropy,Decision tree — Gini,-0.00499,-1.09531,0.33490
4,roc_auc,Decision tree — Entropy,Decision tree — Gini,-0.00386,-0.95311,0.39451


<Figure size 1800x700 with 2 Axes>